In [3]:
import inspect, src.windows as w_
print(w_.__file__)
print("host_names" in inspect.getsource(w_))

c:\Users\HP\Desktop\Ejust\Projects\Data Security\nids\src\windows.py
True


In [5]:
import importlib
import src.windows
importlib.reload(src.windows)
from src.windows import assign_windows, window_table, select_windows, build_tensors

In [6]:
import sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, joblib

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import ClippedScaler
from src.pipeline import DATA, RESULTS
from src.windows import assign_windows, window_table, select_windows, build_tensors

L, SEED = 16, 11
d = joblib.load(RESULTS / "scaler_v2.joblib")
cols, scaler = d["cols"], d["scaler"]
OUT = DATA / "windows"
OUT.mkdir(exist_ok=True)

plan = {
    "train": dict(max_benign=100_000),
    "val":   dict(max_benign=30_000),
    "tune":  dict(max_benign=30_000, cap_per_type=8_000),
    "test":  dict(max_benign=50_000, cap_per_type=10_000),
}

for split, kw in plan.items():
    t0 = time.time()
    meta = pd.read_parquet(DATA / f"{split}.parquet", columns=["row_id", "IPV4_SRC_ADDR", "Attack"])
    m, host_names, atk_names = assign_windows(meta, L)
    del meta; gc.collect()

    w = window_table(m, host_names, atk_names)
    sel = select_windows(w, seed=SEED, **kw)
    X, mask = build_tensors(split, m, sel, cols, scaler, L)
    ws = w.loc[sel]

    np.savez_compressed(OUT / f"{split}.npz", X=X, mask=mask, win_ids=sel,
                        label=ws["label"].to_numpy(),
                        attack_type=ws["attack_type"].astype(str).to_numpy(),
                        n_flows=ws["n_flows"].to_numpy(),
                        first_row_id=ws["first_row_id"].to_numpy())
    m[m["win_id"].isin(sel)][["row_id", "win_id", "pos"]].to_parquet(OUT / f"{split}_flowmap.parquet")

    print(f"\n{split}: {X.shape} in {time.time() - t0:.0f}s")
    print("  windows built in total:", len(w), "| selected:", len(sel))
    print("  share of full 16-flow windows:", round(float((ws['n_flows'] == L).mean()), 3))
    print("  windows per type:", ws["attack_type"].value_counts().to_dict())

    del m, w, ws, X, mask; gc.collect()


train: (100000, 16, 28) in 10s
  windows built in total: 228948 | selected: 100000
  share of full 16-flow windows: 0.999
  windows per type: {'Benign': 100000}

val: (30000, 16, 28) in 3s
  windows built in total: 33760 | selected: 30000
  share of full 16-flow windows: 0.997
  windows per type: {'Benign': 30000}

tune: (40551, 16, 28) in 4s
  windows built in total: 56011 | selected: 40551
  share of full 16-flow windows: 0.997
  windows per type: {'Benign': 29647, 'scanning': 8000, 'password': 880, 'injection': 776, 'ddos': 618, 'xss': 326, 'dos': 292, 'mitm': 12}

test: (109518, 16, 28) in 14s
  windows built in total: 532756 | selected: 109518
  share of full 16-flow windows: 0.881
  windows per type: {'Benign': 50000, 'ddos': 10000, 'password': 10000, 'injection': 10000, 'dos': 10000, 'xss': 10000, 'scanning': 7608, 'backdoor': 1229, 'mitm': 436, 'ransomware': 245}
